In [2]:
import pandas as pd
import numpy as np

df_causal = pd.read_csv("../data/processed/lr_churn.csv")  
# 剔除无宽带用户
df_causal = df_causal[df_causal["InternetService_No"] == 0].copy()
print(df_causal.columns)

Index(['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure',
       'PhoneService', 'PaperlessBilling', 'MonthlyCharges', 'TotalCharges',
       'Churn', 'MultipleLines_No phone service', 'MultipleLines_Yes',
       'InternetService_Fiber optic', 'InternetService_No',
       'OnlineSecurity_No internet service', 'OnlineSecurity_Yes',
       'OnlineBackup_No internet service', 'OnlineBackup_Yes',
       'DeviceProtection_No internet service', 'DeviceProtection_Yes',
       'TechSupport_No internet service', 'TechSupport_Yes',
       'StreamingTV_No internet service', 'StreamingTV_Yes',
       'StreamingMovies_No internet service', 'StreamingMovies_Yes',
       'Contract_One year', 'Contract_Two year',
       'PaymentMethod_Credit card (automatic)',
       'PaymentMethod_Electronic check', 'PaymentMethod_Mailed check'],
      dtype='object')


In [4]:
# ---------------定义干预、结果变量---------------
# T：是否光纤宽带
df_causal["T"] = df_causal["InternetService_Fiber optic"]
# Y：用户流失标签
Y_col = "Churn"
T_col = "T"

# ---------------筛选协变量：剔除结果、干预变量---------------
exclude_list = [Y_col, T_col, "InternetService_Fiber optic","MonthlyCharges", "TotalCharges"]
cov_cols = [col for col in df_causal.columns if col not in exclude_list]

# ---------------样本校验---------------
print("-"*10 + "干预T分布(0=无宽带/DSL,1=光纤)" + "-"*10)
print(df_causal[T_col].value_counts())
print(df_causal[T_col].value_counts(normalize=True).round(3))
print("\n" + "-"*10 + "流失Y分布" + "-"*10)
print(df_causal[Y_col].value_counts())

print(f"\n协变量数量:{len(cov_cols)}")
print("\n" + "-"*10 + "协变量列表" + "-"*10)
print(cov_cols)
print(f"\n光纤宽带用户占总样本的比例:{df_causal[T_col].mean():.2%}")

----------干预T分布(0=无宽带/DSL,1=光纤)----------
T
1    3096
0    2421
Name: count, dtype: int64
T
1    0.561
0    0.439
Name: proportion, dtype: float64

----------流失Y分布----------
Churn
0    3761
1    1756
Name: count, dtype: int64

协变量数量:27

----------协变量列表----------
['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'PaperlessBilling', 'MultipleLines_No phone service', 'MultipleLines_Yes', 'InternetService_No', 'OnlineSecurity_No internet service', 'OnlineSecurity_Yes', 'OnlineBackup_No internet service', 'OnlineBackup_Yes', 'DeviceProtection_No internet service', 'DeviceProtection_Yes', 'TechSupport_No internet service', 'TechSupport_Yes', 'StreamingTV_No internet service', 'StreamingTV_Yes', 'StreamingMovies_No internet service', 'StreamingMovies_Yes', 'Contract_One year', 'Contract_Two year', 'PaymentMethod_Credit card (automatic)', 'PaymentMethod_Electronic check', 'PaymentMethod_Mailed check']

光纤宽带用户占总样本的比例:56.12%


In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
X = df_causal[cov_cols]
T = df_causal[T_col]

# 训练倾向分模型
ps_model = LogisticRegression(max_iter=2000,random_state=42)
ps_model.fit(X,T)
# 计算倾向性得分P(T=1|X)
df_causal["ps"] = ps_model.predict_proba(X)[:,1]
# 分组统计倾向分
print("-"*10 + "两组倾向性得分统计" + "-"*10)
print(df_causal.groupby("T")["ps"].agg(["mean", "std", "min", "max"]).round(4))
# 倾向分模型AUC
auc = roc_auc_score(T, df_causal["ps"])
print(f"\n倾向得分模型AUC:{auc:4f}")


----------两组倾向性得分统计----------
     mean     std     min     max
T                                
0  0.3252  0.2839  0.0000  0.9686
1  0.7457  0.2080  0.0528  0.9943

倾向得分模型AUC:0.872949


In [6]:
df_causal.head(5)

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,PaperlessBilling,MonthlyCharges,TotalCharges,Churn,...,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,T,ps
0,0,0,1,0,1,0,1,29.85,29.85,0,...,0,0,0,0,0,0,1,0,0,0.003493
1,1,0,0,0,34,1,0,56.95,1889.50,0,...,0,0,0,1,0,0,0,1,0,0.089769
2,1,0,0,0,2,1,1,53.85,108.15,1,...,0,0,0,0,0,0,0,1,0,0.209605
3,1,0,0,0,45,0,0,42.30,1840.75,0,...,0,0,0,1,0,0,0,0,0,0.000153
4,0,0,0,0,2,1,1,70.70,151.65,1,...,0,0,0,0,0,0,1,0,1,0.686324


In [ ]:
# ---------------最近邻匹配---------------
from sklearn.neighbors import NearestNeighbors
treat_df = df_causal[df_causal["T"]==1].copy().reset_index(drop=True)
control_df = df_causal[df_causal["T"]==0].copy().reset_index(drop=True)

# 对照组标记是否已经被匹配
control_df["used"] = False
matched_treat_list = []
matched_control_list = []

# 对每个处理组样本，优先匹配距离最近，还未被标记的对照组
for i, treat_row in treat_df.iterrows():
    ps_t = treat_row["ps"]
    avail_control = control_df[control_df["used"] == False]
    if len(avail_control) == 0:
        break

    knn = NearestNeighbors(n_neighbors=1)
    knn.fit(avail_control[["ps"]])
    dist, idx = knn.kneighbors(np.array([[ps_t]]))
    # 筛除倾向分大于0.05的样本
    max_caliper = 0.05
    if dist[0][0] > max_caliper:
        continue
    # 对已经匹配的样本进行标记
    select_idx = avail_control.index[idx[0][0]]
    control_df.loc[select_idx, "used"] = True

    matched_treat_list.append(treat_row)
    matched_control_list.append(control_df.loc[select_idx])

matched_treat = pd.DataFrame(matched_treat_list).reset_index(drop=True)
matched_control = pd.DataFrame(matched_control_list).reset_index(drop=True)

df_matched = pd.concat([matched_treat,matched_control],axis=0).reset_index(drop=True)

print("-"*10 + "无放回1-1匹配结果" + "-"*10)
print(f"匹配前:处理组={len(treat_df)},对照组={len(control_df)}")
print(f"匹配前:处理组={len(matched_treat)},对照组={len(matched_control)}")
print(f"匹配后总样本:{len(df_matched)}")

In [ ]:
# ---------------平衡性检验---------------
def calculate_smd(df, cov_cols, treat_col="T"):
    res = []
    for col in cov_cols:
        t0 = df[df[treat_col]==0][col]
        t1 = df[df[treat_col]==1][col]
        mean0, mean1 = t0.mean(), t1.mean()
        var0, var1 = t0.var(), t1.var()
        smd = (mean1-mean0)/((var0+var1)/2)**0.5
        res.append({"feature":col, "smd":smd})
smd_before = calculate_smd(df_causal,cov_cols)
smd_after = calculate_smd(df_matched,cov_cols)

print("-"*10 + "匹配前SMD" + "-"*10)
print(smd_before)
print("-"*10 + "匹配后SMD" + "-"*10)
print(smd_after)
